## E3SM Monthly Output Analysis
Loc completed an E3SM model run over the Western US using NLDAS-2 forcing data. This notebook analyzes the monthly SWE output, comparing it to SNOTEL observations at a selected site.

In [6]:
import pandas as pd
from buildforcing.datasets import PNNLSnotel

In [2]:
# Load SNOTEL_PATH and E3SM_OUTPUT from environment variables
import os
import sys
SNOTEL_PATH = os.getenv('SNOTEL_PATH')
E3SM_OUTPUT_PATH = os.getenv('E3SM_OUTPUT_PATH')
if not SNOTEL_PATH or not E3SM_OUTPUT_PATH:
    print("SNOTEL_PATH or E3SM_OUTPUT_PATH environment variable is not set.")
    sys.exit(1)

In [4]:
# Read in model output
model_snow = pd.read_csv(E3SM_OUTPUT_PATH + 'SWE_all_locations.csv', parse_dates=['Date'], index_col='Date')

In [31]:
# Function to calculate RMSE from monthly output
def calculate_rmse(site):
    """
    Calculate RMSE between E3SM model output and SNOTEL observations for a given site.
    
    Parameters:
    -----------
    site : str
        Site identifier matching column name in model_snow DataFrame
        
    Returns:
    --------
    dict : Dictionary containing site name, RMSE value, bias, and number of valid observations
    """
    try:
        # Load SNOTEL data for site
        snotel_site_name = site.replace('_', ' ')
        snotel = PNNLSnotel(snotel_site_name, storage_path=SNOTEL_PATH)
        snotel.load_data()
        
        # Resample SNOTEL data to monthly, tracking missing values
        # Only calculate mean for months with 10 or fewer missing values
        swe_series = snotel.data['swe_mm']
        swe_series.index = swe_series.index.tz_localize(None)
        
        # Group by month and calculate missing count and mean
        monthly_groups = swe_series.resample('MS')
        missing_counts = monthly_groups.apply(lambda x: x.isna().sum())
        monthly_means = monthly_groups.mean()
        
        # Set months with >10 missing values to NaN
        snotel_monthly = monthly_means.copy()
        snotel_monthly.loc[missing_counts > 10] = pd.NA
        
        # Get model data for this site
        model_data = model_snow[site].copy()
        
        # Normalize E3SM dates to first of month for alignment
        model_data.index = pd.DatetimeIndex(model_data.index).to_period('M').to_timestamp()
        
        # Align the two datasets (inner join on dates)
        combined = pd.DataFrame({
            'model': model_data,
            'observed': snotel_monthly
        }).dropna()
        
        # Calculate RMSE and bias
        if len(combined) > 0:
            rmse = ((combined['model'] - combined['observed']) ** 2).mean() ** 0.5
            bias = (combined['model'] - combined['observed']).mean()
            mean_swe = combined['observed'].mean()
            n_obs = len(combined)
            n_years = n_obs / 12
            return {'site': site, 'rmse': rmse, 'bias': bias, 'mean_swe': mean_swe, 'n_observations': n_obs, 'n_years': n_years}
        else:
            return {'site': site, 'rmse': None, 'bias': None, 'mean_swe': None, 'n_observations': 0, 'n_years': 0}
            
    except Exception as e:
        print(f"Error processing site {site}: {str(e)}")
        return {'site': site, 'rmse': None, 'bias': None, 'mean_swe': None, 'n_observations': 0, 'n_years': 0, 'error': str(e)}

In [16]:
# Testing
quemazon = PNNLSnotel('Quemazon', storage_path=SNOTEL_PATH)
quemazon.load_data()
quemazon.data.head()

,T_max_C,T_min_C,T_avg_C,precip_mm,swe_mm
Date,,,,,
1980-06-04 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-05 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-06 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-07 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-08 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN


In [32]:
# Test the function on Quemazon
result = calculate_rmse('Quemazon')
print(result)

{'site': 'Quemazon', 'rmse': np.float64(33.656734968867255), 'bias': np.float64(-16.54350971347842), 'mean_swe': np.float64(44.145052085846906), 'n_observations': 216, 'n_years': 18.0}


In [33]:
result = calculate_rmse('Forestdale_Creek')
print(result)

{'site': 'Forestdale_Creek', 'rmse': np.float64(212.40430843540003), 'bias': np.float64(-125.74125075728918), 'mean_swe': np.float64(283.93024206848503), 'n_observations': 183, 'n_years': 15.25}


In [22]:
# Get all site names from model output columns
sites = model_snow.columns.tolist()
print(f"Found {len(sites)} sites in model output")

Found 782 sites in model output


In [34]:
# Calculate RMSE for each site
results = []
for i, site in enumerate(sites, 1):
    print(f"\rProcessing {i}/{len(sites)}: {site[:40]:<40}", end='', flush=True)
    result = calculate_rmse(site)
    results.append(result)

print("\n")  # Move to new line when done

# Create results dataframe
results_df = pd.DataFrame(results)
print(f"Processed {len(results_df)} sites")
results_df

Processing 25/782: Rubicon__2                              Error processing site Rubicon__2: Site name Rubicon  2 not found in the PNNL Snotel metadata file.
Processing 28/782: Squaw_Valley_G_C_                       Error processing site Squaw_Valley_G_C_: Site name Squaw Valley G C  not found in the PNNL Snotel metadata file.
Processing 30/782: Truckee__2                              Error processing site Truckee__2: Site name Truckee  2 not found in the PNNL Snotel metadata file.
Processing 32/782: Ward_Creek__3                           Error processing site Ward_Creek__3: Site name Ward Creek  3 not found in the PNNL Snotel metadata file.
Processing 74/782: Cascade__2                              Error processing site Cascade__2: Site name Cascade  2 not found in the PNNL Snotel metadata file.
Processing 80/782: Culebra__2                              Error processing site Culebra__2: Site name Culebra  2 not found in the PNNL Snotel metadata file.
Processing 108/782: Overland_Res

,site,rmse,bias,mean_swe,n_observations,n_years,error
0,Forestdale_Creek,212.404308,-125.741251,283.930242,183,15.250000,NaN
1,Horse_Meadow,82.964004,-42.224436,122.707259,123,10.250000,NaN
2,Burnside_Lake,63.934598,-31.358075,199.963069,195,16.250000,NaN
3,Summit_Meadow,55.677242,-31.787642,100.512534,96,8.000000,NaN
4,Carson_Pass,136.200060,-74.725528,251.335935,159,13.250000,NaN
...,...,...,...,...,...,...,...
777,Cole_Canyon,52.040724,-29.363180,37.169551,228,19.000000,NaN
778,Taylor_Green,160.681404,-93.949861,137.235443,223,18.583333,NaN
779,Hand_Creek,57.305851,-30.079824,76.280573,228,19.000000,NaN
780,Santiam_Jct_,NaN,NaN,NaN,0,0.000000,Site name Santiam Jct not found in the PNNL S...


In [35]:
# Display summary statistics
print("RMSE Summary Statistics:")
print(results_df['rmse'].describe())
print(f"\nSites with valid RMSE: {results_df['rmse'].notna().sum()}")
print(f"Sites with errors: {results_df['rmse'].isna().sum()}")

RMSE Summary Statistics:
count     730.000000
mean      138.156245
std       121.617361
min         1.634231
25%        60.591147
50%        98.698337
75%       167.603918
max      1134.387640
Name: rmse, dtype: float64

Sites with valid RMSE: 730
Sites with errors: 52


In [36]:
# Optional: Save results to CSV
results_df.to_csv('e3sm_snotel_rmse_comparison.csv', index=False)
print("Results saved to e3sm_snotel_rmse_comparison.csv")

Results saved to e3sm_snotel_rmse_comparison.csv
